In [1]:
import os
import json
from IPython.display import display, Markdown
from dotenv import load_dotenv
load_dotenv('/Users/joe.constantino/Desktop/playground/tableau_langchain/.env')

#base langchain library imports
from langchain_openai import ChatOpenAI
import langgraph
from langgraph.prebuilt import create_react_agent
from langchain.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

#tableau_langchain imports
from utilities.tableau.readMetadata import read
from utilities.tableau.setPrompt import instantiate_prompt
from tools.tableau.getDataTool import get_data

In [2]:
#read in the environment varibales to query a Tableau Published Data Source from a target Tableau Cloud Site or Server Instance
datasource = os.getenv('DATASOURCE_LUID')
read_metadata_url = os.getenv('READ_METADATA')
query_datasource = os.getenv('QUERY_DATASOURCE')
token = os.getenv('AUTH_TOKEN')

In [3]:
## Use the read module to pull in the necessary metadata from your target Tableau Published Data Source. This will module also augment the
## base metadata provided by the read_metadata endpoint by removing unneccesary fields and adding sample field values to each string field.
metadata = read(read_url=read_metadata_url, datasource_luid=datasource, auth_secret=token)

# use the instaniate_prompt module to add the data model metadata to the nlq_to_vds prompt
ready_prompt = instantiate_prompt(metadata = metadata)

reading in your field metadata...
looking up field values...
prompt is ready!


In [4]:
#Langchain Example
# Initialize an LLM and an agent with the get_data tool to answer natural language questions with Tableau Published Datasources
llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
tools = [get_data]

tableauHeadlessAgent = initialize_agent(
    tools,
    llm,
    agent=AgentType.OPENAI_FUNCTIONS, # Use OpenAI's function calling
    verbose = False
)

# Run the agent
response = tableauHeadlessAgent.invoke("which states sell the most?")
display(Markdown(response['output']))

/var/folders/3t/f2z2c6_n3951jmt86v5rvgp80000gp/T/ipykernel_62252/2251018611.py:6: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. See LangGraph documentation for more details: https://langchain-ai.github.io/langgraph/. Refer here for its pre-built ReAct agent: https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/
  tableauHeadlessAgent = initialize_agent(


The states that sell the most based on sales are:

1. California: $457,687.63
2. New York: $310,876.27
3. Washington: $138,641.27
4. Texas: $170,188.05
5. Pennsylvania: $116,511.91

These states have the highest sales amounts.

In [5]:
#Langgraph Example
# Initialize an LLM and an agent with the get_data tool to answer natural language questions with Tableau Published Datasources
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
tools = [get_data]

tableauHeadlessAgent = create_react_agent(model, tools)

# Run the agent
messages = tableauHeadlessAgent.invoke({"messages": [("human",'which products are my top customers buying?')]})
display(Markdown(messages['messages'][3].content))

Here are the top products bought by your top customers along with their sales amounts:

1. **Hunter Lopez**: 
   - Product: Canon imageCLASS 2200 Advanced Copier
   - Sales: $10,499.97

2. **Harry Marie**: 
   - Product: Canon PC1060 Personal Laser Copier
   - Sales: $4,899.93

3. **Greg Guthrie**: 
   - Product: Staples
   - Sales: $5,199.96

4. **Raymond Buch**: 
   - Product: Canon imageCLASS 2200 Advanced Copier
   - Sales: $13,999.96

5. **Todd Sumrall**: 
   - Product: 3D Systems Cube Printer, 2nd Generation, Magenta
   - Sales: $5,199.96

6. **Sanjit Chand**: 
   - Product: Ibico EPK-21 Electric Binding System
   - Sales: $9,449.95

7. **Tamara Chand**: 
   - Product: Canon imageCLASS 2200 Advanced Copier
   - Sales: $17,499.95

8. **Tom Ashbrook**: 
   - Product: Canon imageCLASS 2200 Advanced Copier
   - Sales: $11,199.97

9. **Adrian Barton**: 
   - Product: GBC Ibimaster 500 Manual ProClick Binding System
   - Sales: $9,892.74

10. **Bill Shonely**: 
    - Product: 3D Systems Cube Printer, 2nd Generation, Magenta
    - Sales: $9,099.93

11. **Karen Daniels**: 
    - Product: Ativa V4110MDD Micro-Cut Shredder
    - Sales: $4,899.93

12. **Sean Miller**: 
    - Product: Cisco TelePresence System EX90 Videoconferencing Unit
    - Sales: $22,638.48

These products represent significant sales from your top customers.